# Spiking Neural Network (SNN) - TensorFlow / Keras

**Goal:** Classify rate-coded digits with leaky integrate-and-fire neurons.

This notebook favors clear, production-style structure: seeded runs,
explicit data preparation, small reusable modules, and compact
training loops that can be expanded for larger experiments.


## Architecture Notes

- **What it learns:** Membrane voltage integrates input over time and emits thresholded spikes.
- **Where it is used:** neuromorphic computing, event sensors, and low-power temporal processing.
- **Why it works:** the architecture builds a useful bias into the computation, so the model does not need to rediscover that structure from data alone.
- **Output to expect:** classification models return class scores/probabilities, reconstruction models return reconstructed inputs, and generative models return new or denoised samples.


## Visual Intuition

Run this cell before or after training. It is lightweight and framework-independent, so it explains the network idea without requiring a long training run.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
fig.suptitle("Spiking Neural Network: intuition, signal flow, and output", fontsize=13)
axes[0].axis("off")
layers = ['rates', 'spikes', 'readout']
xs = np.linspace(0.1, 0.9, len(layers))
for xpos, label in zip(xs, layers):
    axes[0].scatter([xpos], [0.55], s=1200, color="#4C78A8", alpha=0.18, edgecolors="#4C78A8")
    axes[0].text(xpos, 0.55, label, ha="center", va="center", fontsize=9)
for a, b in zip(xs[:-1], xs[1:]):
    axes[0].annotate("", xy=(b - 0.045, 0.55), xytext=(a + 0.045, 0.55), arrowprops=dict(arrowstyle="->", lw=1.5))
axes[0].set_title("How data moves")
x = np.linspace(-3, 3, 160)
y = (np.sin(5*x)>0).astype(float)
axes[1].plot(x, y, color="#F58518", lw=2)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_title("Toy behavior")
axes[1].grid(alpha=0.25)
values = np.array([.10,.90])
axes[2].bar(range(len(values)), values, color=["#54A24B", "#E45756", "#72B7B2", "#B279A2"][:len(values)])
axes[2].set_title("Typical output")
axes[2].set_xticks(range(len(values)))
axes[2].set_xticklabels(['quiet', 'spike'])
axes[2].grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(SEED)
print(f"TensorFlow: {tf.__version__}")
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X = (digits.data / 16.0).astype("float32")
y = digits.target.astype("int64")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(1024).batch(64)


In [ ]:
@tf.custom_gradient
def surrogate_spike(membrane):
    spikes = tf.cast(membrane > 1.0, tf.float32)
    def grad(upstream):
        return upstream / tf.square(1 + 10 * tf.abs(membrane - 1.0))
    return spikes, grad


class SpikingMLP(keras.Model):
    def __init__(self, steps=12):
        super().__init__()
        self.steps = steps
        self.fc1 = layers.Dense(128)
        self.fc2 = layers.Dense(10)

    def call(self, rates, training=False):
        membrane = tf.zeros((tf.shape(rates)[0], 128))
        output_sum = tf.zeros((tf.shape(rates)[0], 10))
        for _ in range(self.steps):
            spikes_in = tf.cast(tf.random.uniform(tf.shape(rates)) < rates, tf.float32)
            membrane = 0.9 * membrane + self.fc1(spikes_in)
            spikes = surrogate_spike(membrane)
            membrane = membrane * (1 - spikes)
            output_sum += self.fc2(spikes)
        return output_sum / self.steps


model = SpikingMLP()
optimizer = keras.optimizers.AdamW(1e-3)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)


In [ ]:
for epoch in range(10):
    for batch_x, batch_y in train_ds:
        with tf.GradientTape() as tape:
            loss = loss_fn(batch_y, model(batch_x, training=True))
        optimizer.apply_gradients(zip(tape.gradient(loss, model.trainable_variables), model.trainable_variables))
    logits = model(tf.constant(X_test), training=False)
    accuracy = tf.reduce_mean(tf.cast(tf.equal(tf.cast(tf.argmax(logits, axis=1), tf.int64), tf.constant(y_test, dtype=tf.int64)), tf.float32))
    print(f"epoch={epoch+1:02d} accuracy={float(accuracy):.3f}")
